# RelCheck — BLIP-2 Enrichment Test

**Goal:** Test RelCheck enrichment on real BLIP-2 captions (short, ~10-30 words).
Uses `enrich_caption_v3` which auto-dispatches to enrichment mode for short captions.

**Filtering:** Excludes nonsense captions (repetition, <5 words, gibberish) before evaluation.

**KB:** Reuses knowledge bases from the 600-image run (no rebuild).

**Evaluation:**
1. R-POPE LLM-Judge (Llama-3.3-70B, text-only, measures caption quality directly)
2. Error analysis + qualitative examples


In [ ]:
# ============================================================
# Cell 1 — Install + Setup
# ============================================================
!pip install -q together gdown transformers pillow torch torchvision accelerate bitsandbytes>=0.46.1
!pip install -q spacy open-clip-torch json-repair pysbd rapidfuzz nltk
!python -m spacy download en_core_web_sm -q
!pip install -q nltk
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import os, sys, json, time, random, gc
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import base64

from google.colab import drive
drive.mount("/content/drive")

# ── Clone / pull relcheck_v2 ──────────────────────────────
REPO_DIR = "/content/RelCheck"
if not os.path.exists(f"{REPO_DIR}/.git"):
    import shutil
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.system(f"git clone https://github.com/siddhipatil503/RelCheck {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Settings ─────────────────────────────────────────────
TOGETHER_API_KEY = ""          # <-- paste your Together.ai key
N_IMAGES         = 50          # 50 = fast validation; 200 = publication quality
RANDOM_SEED      = 42

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH      = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"
SAVE_DIR_600     = "/content/drive/MyDrive/RelCheck_Data/run_600"  # reuse KB + captions from here
SAVE_DIR         = "/content/drive/MyDrive/RelCheck_Data/blip2_test"  # save results here

VLM_MODEL    = "Qwen/Qwen3-VL-8B-Instruct"
LLM_MODEL    = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
GDINO_ID     = "IDEA-Research/grounding-dino-tiny"
VITPOSE_ID   = "usyd-community/vitpose-base-simple"
DETECTION_THRESHOLD = 0.25

os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
random.seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Init relcheck_v2 API client ──────────────────────────
import relcheck_v2.api as _api
_api.init_client(TOGETHER_API_KEY)

def save_checkpoint(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_checkpoint(path):
    if os.path.exists(path): 
        with open(path) as f: return json.load(f)
    return None

print(f"Device : {DEVICE}")
print(f"Save to: {SAVE_DIR}")
print(f"Images : {N_IMAGES}")


In [ ]:
# ============================================================
# Cell 2 — Load Models (GroundingDINO + ViTPose)
# ============================================================
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    VitPoseForPoseEstimation,
)

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO ready.")

print("Loading ViTPose...")
vitpose_processor = AutoProcessor.from_pretrained(VITPOSE_ID)
vitpose_model = VitPoseForPoseEstimation.from_pretrained(VITPOSE_ID).to(DEVICE)
print("  ViTPose ready.")

# Inject into relcheck_v2.models singletons so get_gdino() reuses them
import relcheck_v2.models as _models
_models._gdino_model     = gdino_model
_models._gdino_processor = gdino_processor
_models.DEVICE           = DEVICE

# Also set detection threshold in config
import relcheck_v2.config as _cfg
_cfg.DETECTION_THRESHOLD = DETECTION_THRESHOLD

print(f"\nAll models ready on {DEVICE}.")


In [ ]:
# ============================================================
# Cell 3 — Load Images + R-Bench Data
# ============================================================
import pathlib, zipfile

# ── Download R-Bench if needed ──
if not os.path.exists(RBENCH_PATH):
    print("Downloading R-Bench...")
    RBENCH_DIR = "/content/R-Bench"
    if not os.path.exists(RBENCH_DIR):
        os.system(f"git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}")
    ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
    if not os.path.exists(f"{RBENCH_DIR}/data"):
        os.system(f"gdown --id 1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH -O {ANNOTATIONS_ZIP} -q")
        with zipfile.ZipFile(ANNOTATIONS_ZIP) as z:
            z.extractall(RBENCH_DIR)
    for f in sorted(pathlib.Path(RBENCH_DIR).rglob("*.json")):
        try:
            data = json.load(open(f))
            if isinstance(data, list) and len(data) > 50:
                normalized = []
                for entry in data:
                    img_file = entry.get("image") or entry.get("img","")
                    img_id   = os.path.splitext(os.path.basename(img_file))[0]
                    q = entry.get("text") or entry.get("question","")
                    a = (entry.get("label") or entry.get("answer","")).lower().strip()
                    if img_id and q and a in ("yes","no"):
                        normalized.append({"img_id": img_id, "question": q, "answer": a})
                if len(normalized) > 100:
                    with open(RBENCH_PATH, "w") as fout:
                        json.dump(normalized, fout)
                    print(f"  Saved {len(normalized)} Q&A pairs.")
                    break
        except: pass

rbench = json.load(open(RBENCH_PATH))

def _get_field(entry, *keys, default=None):
    for k in keys:
        if k in entry: return entry[k]
    return default

img_to_questions = defaultdict(list)
for entry in rbench:
    # Handle both normalized (img_id) and raw (image/img) formats
    img_id = _get_field(entry, "img_id", "img", "image")
    if img_id:
        # Strip path + extension if it's a filename
        img_id = os.path.splitext(os.path.basename(img_id))[0]
    q  = _get_field(entry, "question", "text", "q")
    gt = (_get_field(entry, "answer", "label", "gt") or "").lower().strip()
    if img_id and q and gt in ("yes", "no"):
        img_to_questions[img_id].append({"question": q, "answer": gt})

# ── Load images — only from IDs that have cached LLaVA captions + KB ──
LLAVA_CAPS_PATH = f"{SAVE_DIR}/llava_captions.json"
KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"
_cached_llava = json.load(open(LLAVA_CAPS_PATH)) if os.path.exists(LLAVA_CAPS_PATH) else {}
_cached_kb    = json.load(open(KB_PATH)) if os.path.exists(KB_PATH) else {}

all_rbench_ids = sorted({os.path.splitext(os.path.basename(_get_field(e, "img_id", "img", "image", default="")))[0] for e in rbench} - {""})

# Filter to IDs that have cached LLaVA captions AND KB (skip re-computing)
if _cached_llava:
    available_ids = [i for i in all_rbench_ids if i in _cached_llava and i in _cached_kb]
    print(f"R-Bench IDs: {len(all_rbench_ids)}, with cached LLaVA+KB: {len(available_ids)}")
else:
    available_ids = all_rbench_ids
    print(f"No cached LLaVA captions found — will generate from scratch for {len(available_ids)} IDs")

# ── Use saved image IDs from KB Builder if available (ensures consistency) ──
KB_IMAGE_IDS_DIR = "/content/drive/MyDrive/RelCheck_Data/knowledge_bases"
NOCAPS_IDS_PATH = f"{KB_IMAGE_IDS_DIR}/nocaps_image_ids.json"
COCO_IDS_PATH   = f"{KB_IMAGE_IDS_DIR}/coco_image_ids.json"

if os.path.exists(NOCAPS_IDS_PATH) or os.path.exists(COCO_IDS_PATH):
    kb_ids = []
    if os.path.exists(NOCAPS_IDS_PATH):
        kb_ids += json.load(open(NOCAPS_IDS_PATH))
    if os.path.exists(COCO_IDS_PATH):
        kb_ids += json.load(open(COCO_IDS_PATH))
    # Filter to IDs that exist in R-Bench and available images
    selected_ids = [i for i in kb_ids if i in available_ids][:N_IMAGES]
    print(f"Using {len(selected_ids)} image IDs from KB Builder (consistent with KB).")
else:
    random.shuffle(available_ids)
    selected_ids = available_ids[:N_IMAGES]
    print(f"No KB Builder ID lists found — random sampling {len(selected_ids)} images.")

images = {}
for img_id in selected_ids:
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(DRIVE_IMAGES_DIR, img_id + ext)
        if os.path.exists(p):
            images[img_id] = Image.open(p).convert("RGB")
            break

print(f"Loaded {len(images)} images, {sum(len(q) for q in img_to_questions.values())} total Q&A pairs")
print(f"Avg questions per image: {np.mean([len(img_to_questions[i]) for i in images]):.1f}")


In [ ]:
# ============================================================
# Cell 4 — BLIP-2 Captioning + Nonsense Filter
# ============================================================
# Load BLIP-2 captions from 600-image run (no regeneration needed).
# Then filter out nonsense captions: repetitions, too short, gibberish.

import re

BLIP2_CAPS_PATH = f"{SAVE_DIR_600}/captions.json"
blip2_captions_raw = load_checkpoint(BLIP2_CAPS_PATH) or {}

# Filter to only images we loaded
blip2_captions_raw = {k: v for k, v in blip2_captions_raw.items() if k in images}
print(f"Loaded {len(blip2_captions_raw)} BLIP-2 captions from 600-image run.")

# ── Nonsense Detection ──────────────────────────────────
def is_nonsense(caption):
    """Detect nonsense captions: repetitions, too short, gibberish.
    Returns (is_bad: bool, reason: str)."""
    if not caption or not caption.strip():
        return True, "empty"
    
    words = caption.lower().split()
    
    # Too short (< 5 words)
    if len(words) < 5:
        return True, f"too_short ({len(words)} words)"
    
    # Repeating words: if any word appears > 40% of the time
    word_counts = Counter(words)
    for word, count in word_counts.items():
        if len(word) >= 3 and count / len(words) > 0.4:
            return True, f"repeating_word (\"{word}\" x{count}/{len(words)})"
    
    # Repeating phrases: check for repeated 3-grams
    if len(words) >= 6:
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        for tri, count in trigram_counts.items():
            if count >= 3:
                return True, f"repeating_phrase (\"{' '.join(tri)}\" x{count})"
    
    # Repeating bigrams (even more sensitive)
    if len(words) >= 4:
        bigrams = [tuple(words[i:i+2]) for i in range(len(words)-1)]
        bigram_counts = Counter(bigrams)
        for bi, count in bigram_counts.items():
            if count >= 4:
                return True, f"repeating_bigram (\"{' '.join(bi)}\" x{count})"
    
    # Mostly non-alphabetic characters
    alpha_ratio = sum(c.isalpha() or c.isspace() for c in caption) / max(len(caption), 1)
    if alpha_ratio < 0.6:
        return True, f"gibberish (alpha_ratio={alpha_ratio:.2f})"
    
    return False, "ok"

# Apply filter
blip2_captions = {}
filtered_out = []
for img_id, cap in blip2_captions_raw.items():
    bad, reason = is_nonsense(cap)
    if bad:
        filtered_out.append((img_id, reason, cap[:80]))
    else:
        blip2_captions[img_id] = cap

print(f"\nAfter filtering: {len(blip2_captions)} clean captions "
      f"({len(filtered_out)} removed)")

if filtered_out:
    print(f"\nFiltered out ({len(filtered_out)}):")
    for img_id, reason, preview in filtered_out[:15]:
        print(f"  {img_id}: [{reason}] {preview}...")

# Update images dict to only include images with clean captions
images = {k: v for k, v in images.items() if k in blip2_captions}
print(f"\nImages for evaluation: {len(images)}")

words = [len(c.split()) for c in blip2_captions.values()]
print(f"Avg caption length: {np.mean(words):.0f} words "
      f"(min={min(words)}, max={max(words)})")
print(f"Captions < 30 words: {sum(1 for w in words if w < 30)}/{len(words)} → enrichment mode")
print(f"Captions >= 30 words: {sum(1 for w in words if w >= 30)}/{len(words)} → correction mode")


In [ ]:
# ============================================================
# Cell 5 — Knowledge Base: Load from 600-run + Gap-Fill Missing
# ============================================================
# 1. Load cached KB from 600-image run
# 2. Build KB for any missing images using:
#    - relcheck_v2.detection.detect_objects (batched, 4 labels/call)
#    - Qwen VLM for relationship descriptions
# 3. Save combined KB

# NOTE: The 600-image KB was built with ALL labels in a single DINO call,
# which causes attention dilution and inaccurate detections. If results
# look noisy, delete the cached KB and rebuild fully with this cell.

from relcheck_v2.detection import detect_objects as detect_objects_batched, dedup_detections
from relcheck_v2.api import vlm_call

KB_600_PATH = f"{SAVE_DIR_600}/knowledge_bases.json"
KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"

BROAD_CATEGORIES = [
    "person","man","woman","child","boy","girl","dog","cat","bird","horse","animal",
    "car","bicycle","motorcycle","bus","truck","chair","table","bench","couch","bed",
    "food","plate","bowl","cup","bottle","glass","bag","umbrella","phone","book","sign",
    "hat","jacket","vest","helmet","glasses","tree","flower","grass","water",
]

VLM_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe:
- SPATIAL: Where are they relative to each other?
- ACTIONS: What is each person/animal doing?
- ATTRIBUTES: Relevant attributes tied to relationships

Rules: only describe what you can clearly see. Be brief and factual.
Format as a numbered list."""

def compute_spatial_facts(dets):
    """Compute pairwise spatial relationships from bounding boxes."""
    # Build label → best bbox map
    bbox_map = {}
    for det in sorted(dets, key=lambda x: -x.score):
        if det.label not in bbox_map:
            bbox_map[det.label] = det.bbox

    facts = []
    items = list(bbox_map.items())
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            la, ba = items[i]; lb, bb = items[j]
            cx_a, cy_a = (ba[0]+ba[2])/2, (ba[1]+ba[3])/2
            cx_b, cy_b = (bb[0]+bb[2])/2, (bb[1]+bb[3])/2
            dy = cy_a - cy_b
            dx = cx_a - cx_b
            if abs(dy) > 0.1:
                rel = "below" if dy > 0 else "above"
                facts.append(f"{la} is {rel} {lb}")
            elif abs(dx) > 0.1:
                rel = "to the right of" if dx > 0 else "to the left of"
                facts.append(f"{la} is {rel} {lb}")
    return facts

def build_kb_for_image(img_id, image):
    """Build visual KB using batched DINO (4 labels/call) + Qwen VLM."""
    # Batched detection — 4 labels per forward pass (avoids attention dilution)
    raw_dets = detect_objects_batched(image, BROAD_CATEGORIES, batch_size=4)
    dets = dedup_detections(raw_dets)

    # Cap at top-20 most confident
    dets_sorted = sorted(dets, key=lambda x: -x.score)[:20]

    labels = list({d.label for d in dets_sorted})
    spatial_facts = compute_spatial_facts(dets_sorted)

    # VLM description via Qwen
    visual_description = ""
    if labels:
        buf = BytesIO()
        image.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        det_list = "\n".join(f"- {l}" for l in labels[:15])
        raw = vlm_call([{"role": "user", "content": [
            {"type": "text", "text": VLM_KB_PROMPT.format(detection_list=det_list)},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}], max_tokens=300)
        visual_description = raw or ""

    return {
        "detections": [{"label": d.label, "score": d.score, "bbox": d.bbox} for d in dets_sorted],
        "spatial_facts": spatial_facts,
        "hard_facts": [f"{l} is present" for l in labels],
        "visual_description": visual_description,
    }

# ── Step 1: Load from 600-image run ──
knowledge_bases = load_checkpoint(KB_600_PATH) or {}
n_from_600 = sum(1 for i in images if i in knowledge_bases)
print(f"Loaded KB from 600-image run: {n_from_600}/{len(images)} images covered.")

# ── Step 2: Also check local cache ──
local_kb = load_checkpoint(KB_PATH) or {}
for img_id in images:
    if img_id not in knowledge_bases and img_id in local_kb:
        knowledge_bases[img_id] = local_kb[img_id]

n_cached = sum(1 for i in images if i in knowledge_bases)
print(f"After local cache: {n_cached}/{len(images)} images covered.")

# ── Step 3: Gap-fill missing images (batched DINO + Qwen) ──
missing = [i for i in images if i not in knowledge_bases]
if missing:
    print(f"\nBuilding KB for {len(missing)} missing images "
          f"(batched DINO 4-labels/call + Qwen VLM)...")
    for idx, img_id in enumerate(missing):
        t0 = time.time()
        knowledge_bases[img_id] = build_kb_for_image(img_id, images[img_id])
        elapsed = time.time() - t0
        kb = knowledge_bases[img_id]
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f"  [{idx+1}/{len(missing)}] {img_id}: "
                  f"{len(kb['detections'])} dets, {len(kb['spatial_facts'])} spatial ({elapsed:.1f}s)")
            save_checkpoint(knowledge_bases, KB_PATH)
        time.sleep(0.3)
    save_checkpoint(knowledge_bases, KB_PATH)
    print(f"Gap-fill complete. Saved to {KB_PATH}")
else:
    print("All images have KB — no gap-fill needed.")

# ── Optional: Rebuild ALL KBs with batched detection ──
# Uncomment the next line to force rebuild (deletes cached KB from 600-run)
# REBUILD_ALL = True
REBUILD_ALL = False

if REBUILD_ALL:
    print(f"\n⚠ REBUILDING all {len(images)} KBs with batched detection...")
    for idx, (img_id, img) in enumerate(images.items()):
        t0 = time.time()
        knowledge_bases[img_id] = build_kb_for_image(img_id, img)
        elapsed = time.time() - t0
        if (idx + 1) % 10 == 0 or idx == 0:
            kb = knowledge_bases[img_id]
            print(f"  [{idx+1}/{len(images)}] {img_id}: "
                  f"{len(kb['detections'])} dets ({elapsed:.1f}s)")
            save_checkpoint(knowledge_bases, KB_PATH)
    save_checkpoint(knowledge_bases, KB_PATH)
    print("Rebuild complete.")

print(f"\nKB ready: {sum(1 for i in images if i in knowledge_bases)}/{len(images)} images.")


In [ ]:
# ============================================================
# Cell 6 — RelCheck Enrichment (enrich_caption_v3)
# ============================================================
# Uses enrich_caption_v3 which auto-dispatches:
#   - Short captions (<30 words): KB-guided enrichment (most BLIP-2 captions)
#   - Long captions (>=30 words): surgical correction (rare for BLIP-2)
#
# variant_a: enrichment only (include_addendum=False) ← isolates error correction
# variant_b: enrichment + KB facts (include_addendum=True) ← full system

from relcheck_v2.correction import enrich_caption_v3

ENRICHED_ONLY_PATH = f"{SAVE_DIR}/blip2_enriched_only.json"
ENRICHED_FULL_PATH = f"{SAVE_DIR}/blip2_enriched_full.json"

def run_relcheck(caps, save_path, include_addendum, label):
    results = load_checkpoint(save_path) or {}
    todo = [i for i in images if i not in results and caps.get(i)]
    if not todo:
        print(f"{label}: loaded {len(results)} from cache.")
        return results

    print(f"{label}: enriching {len(todo)} captions (addendum={include_addendum})...")
    for idx, img_id in enumerate(todo):
        cap = caps[img_id]
        kb  = knowledge_bases.get(img_id, {})
        t0  = time.time()

        result = enrich_caption_v3(
            img_id           = img_id,
            caption          = cap,
            kb               = kb,
            pil_image        = images.get(img_id),
            cross_captions   = None,
            include_addendum = include_addendum,
        )

        results[img_id] = {
            "original"  : result.original,
            "corrected" : result.corrected,
            "status"    : result.status,
            "edit_rate" : result.edit_rate,
            "n_triples" : getattr(result, "n_triples", 0),
            "n_addendum": getattr(result, "n_addendum", 0),
            "errors"    : [{"claim": e.triple.claim, "confidence": e.confidence.value,
                            "reason": e.reason} for e in (result.errors or [])]
                          if hasattr(result, "errors") and result.errors else [],
            "time_s"    : round(time.time() - t0, 1),
        }

        if (idx + 1) % 5 == 0 or (hasattr(result, 'status') and result.status == "modified"):
            r = results[img_id]
            print(f"  [{idx+1}/{len(todo)}] {img_id} → {r['status']} "
                  f"edit={r['edit_rate']:.3f} ({r['time_s']}s)")
            save_checkpoint(results, save_path)

    save_checkpoint(results, save_path)
    return results

enriched_only = run_relcheck(blip2_captions, ENRICHED_ONLY_PATH,
                              include_addendum=False, label="enrichment-only")
enriched_full = run_relcheck(blip2_captions, ENRICHED_FULL_PATH,
                              include_addendum=True,  label="enrichment+addendum")

# ── Diagnostics ──
print(f"\n{'='*65}")
print("PIPELINE DIAGNOSTICS (enrichment-only variant)")
print(f"{'='*65}")
n_modified  = sum(1 for v in enriched_only.values() if v["status"]=="modified")
n_unchanged = sum(1 for v in enriched_only.values() if v["status"]=="unchanged")
n_skipped   = sum(1 for v in enriched_only.values() if v.get("status")=="skipped")
print(f"  Total images     : {len(enriched_only)}")
print(f"  Images modified  : {n_modified}")
print(f"  Images unchanged : {n_unchanged}")
print(f"  Images skipped   : {n_skipped}")

# Word count comparison
orig_words = [len(blip2_captions[i].split()) for i in enriched_only if blip2_captions.get(i)]
enr_words  = [len(v["corrected"].split()) for v in enriched_only.values() if v.get("corrected")]
print(f"\n  Avg original words : {np.mean(orig_words):.0f}")
print(f"  Avg enriched words : {np.mean(enr_words):.0f}")

# ── Summary ──
print(f"\n{'='*65}")
print(f"{'Variant':<25} {'Modified':>10} {'Avg Edit':>10}")
print("-" * 65)
for label, data in [("enrichment-only", enriched_only),
                     ("enrichment+addendum", enriched_full)]:
    n = len(data)
    n_mod = sum(1 for v in data.values() if v["status"] == "modified")
    avg_e = np.mean([v["edit_rate"] for v in data.values()])
    orig_w = np.mean([len(blip2_captions[i].split()) for i in data if blip2_captions.get(i)])
    corr_w = np.mean([len(v["corrected"].split()) for v in data.values() if v.get("corrected")])
    print(f"{label:<25} {n_mod:>4}/{n:<4} ({n_mod/n:.0%})  "
          f"{avg_e:>8.3f}  {orig_w:.0f}→{corr_w:.0f}w")


In [ ]:
# ============================================================
# Cell 7 — R-POPE LLM-Judge Evaluation
# ============================================================
# Evaluates 3 variants (original BLIP-2, enrichment-only, enrichment+addendum)
# using Llama-3.3-70B as a text-only judge (no image access).

RPOPE_PATH = f"{SAVE_DIR}/rpope_blip2.json"

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""

def rpope_judge(caption, question):
    from relcheck_v2.api import llm_call as _llm
    raw = _llm([{"role": "user", "content":
                 RPOPE_PROMPT.format(caption=caption, question=question)}],
               max_tokens=5)
    if raw:
        rl = raw.lower()
        if "yes" in rl and "no" not in rl: return "yes"
        if "no" in rl: return "no"
    return "unknown"

def compute_metrics(pairs):
    tp = sum(1 for p,g in pairs if p=="yes" and g=="yes")
    fp = sum(1 for p,g in pairs if p=="yes" and g=="no")
    fn = sum(1 for p,g in pairs if p=="no"  and g=="yes")
    tn = sum(1 for p,g in pairs if p=="no"  and g=="no")
    n  = len(pairs)
    acc  = (tp+tn)/n if n else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec  = tp/(tp+fn) if (tp+fn) else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1,"n":n}

method_captions = {
    "blip2_orig"          : blip2_captions,
    "blip2_enriched_only" : {i: v["corrected"] for i,v in enriched_only.items()
                              if v.get("corrected")},
    "blip2_enriched_full" : {i: v["corrected"] for i,v in enriched_full.items()
                              if v.get("corrected")},
}

rpope_raw = load_checkpoint(RPOPE_PATH) or {}
methods_todo = []
for m in method_captions:
    sample_id = next(iter(images), None)
    key = f"{m}|||{sample_id}" if sample_id else m
    if key not in rpope_raw:
        methods_todo.append(m)

if methods_todo:
    print(f"Evaluating {len(methods_todo)} methods: {methods_todo}")
else:
    print("All methods cached — loading results.")

for method in methods_todo:
    caps = method_captions[method]
    print(f"\n{method}: evaluating {len(caps)} captions...")
    for idx, img_id in enumerate(caps):
        qs = [e for e in rbench_data if e["img_id"] == img_id]
        for entry in qs:
            q = entry["question"]
            gt = entry["answer"]
            key = f"{method}|||{img_id}|||{q}"
            if key in rpope_raw:
                continue
            pred = rpope_judge(caps[img_id], q)
            rpope_raw[key] = {"method": method, "img_id": img_id,
                              "question": q, "gt": gt, "pred": pred}
        if (idx+1) % 20 == 0:
            save_checkpoint(rpope_raw, RPOPE_PATH)
            print(f"  [{idx+1}/{len(caps)}]")
    save_checkpoint(rpope_raw, RPOPE_PATH)

# ── Aggregate ──
results_by_method = defaultdict(list)
for key, entry in rpope_raw.items():
    m = entry.get("method", key.split("|||")[0])
    results_by_method[m].append((entry["pred"], entry["gt"]))

print(f"\n{'='*70}")
print("R-POPE LLM-JUDGE RESULTS — BLIP-2 Enrichment")
print(f"{'='*70}")
print(f"{'Method':<25} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'N':>6}")
print("-" * 70)

baseline_acc = None
for method in ["blip2_orig", "blip2_enriched_only", "blip2_enriched_full"]:
    if method not in results_by_method:
        continue
    pairs = results_by_method[method]
    m = compute_metrics(pairs)
    delta = ""
    if baseline_acc is None:
        baseline_acc = m["accuracy"]
    else:
        d = m["accuracy"] - baseline_acc
        delta = f"  ({d:+.1%})"
    print(f"{method:<25} {m['accuracy']:>6.1%} {m['precision']:>6.1%} "
          f"{m['recall']:>6.1%} {m['f1']:>6.1%} {m['n']:>5}{delta}")

# ── Per-image analysis: which images improved/regressed? ──
print(f"\n{'='*70}")
print("PER-IMAGE ANALYSIS (enrichment-only vs original)")
print(f"{'='*70}")
img_orig = defaultdict(list)
img_enr  = defaultdict(list)
for key, entry in rpope_raw.items():
    m = entry.get("method")
    correct = (entry["pred"] == entry["gt"])
    if m == "blip2_orig":
        img_orig[entry["img_id"]].append(correct)
    elif m == "blip2_enriched_only":
        img_enr[entry["img_id"]].append(correct)

improved, regressed, unchanged = [], [], []
for img_id in img_orig:
    if img_id not in img_enr: continue
    orig_score = sum(img_orig[img_id]) / len(img_orig[img_id])
    enr_score  = sum(img_enr[img_id])  / len(img_enr[img_id])
    if enr_score > orig_score:
        improved.append((img_id, orig_score, enr_score))
    elif enr_score < orig_score:
        regressed.append((img_id, orig_score, enr_score))
    else:
        unchanged.append(img_id)

print(f"  Improved : {len(improved)}")
print(f"  Regressed: {len(regressed)}")
print(f"  Unchanged: {len(unchanged)}")

if improved:
    print(f"\n  Top improved:")
    for img_id, o, e in sorted(improved, key=lambda x: x[2]-x[1], reverse=True)[:5]:
        print(f"    {img_id}: {o:.0%} → {e:.0%}")
if regressed:
    print(f"\n  Top regressed:")
    for img_id, o, e in sorted(regressed, key=lambda x: x[1]-x[2], reverse=True)[:5]:
        print(f"    {img_id}: {o:.0%} → {e:.0%}")


In [ ]:
# ============================================================
# Cell 8 — Error Analysis + Qualitative Examples
# ============================================================

print(f"{'='*65}")
print("ENRICHMENT BREAKDOWN")
print(f"{'='*65}")
n_unchanged = sum(1 for v in enriched_only.values() if v["status"]=="unchanged")
n_modified  = sum(1 for v in enriched_only.values() if v["status"]=="modified")
n_skipped   = sum(1 for v in enriched_only.values() if v.get("status")=="skipped")
print(f"  unchanged : {n_unchanged}")
print(f"  modified  : {n_modified}")
print(f"  skipped   : {n_skipped}")

# ── Show 5 enriched examples ──
print(f"\n{'='*65}")
print("ENRICHED EXAMPLES (top 5 by edit rate)")
print(f"{'='*65}")
modified = [(i, v) for i, v in enriched_only.items() if v["status"] == "modified"]
modified.sort(key=lambda x: -x[1]["edit_rate"])

for img_id, v in modified[:5]:
    print(f"\n[{img_id}]  edit_rate={v['edit_rate']:.3f}")
    print(f"  ORIGINAL : {v['original']}")
    print(f"  ENRICHED : {v['corrected'][:300]}")

# ── Show regressed examples (if any) ──
if regressed:
    print(f"\n{'='*65}")
    print("REGRESSED EXAMPLES (accuracy dropped after enrichment)")
    print(f"{'='*65}")
    for img_id, orig_score, enr_score in sorted(regressed, key=lambda x: x[1]-x[2], reverse=True)[:5]:
        v = enriched_only.get(img_id, {})
        print(f"\n[{img_id}]  R-POPE: {orig_score:.0%} → {enr_score:.0%}")
        print(f"  ORIGINAL : {v.get('original', '?')}")
        print(f"  ENRICHED : {v.get('corrected', '?')[:300]}")


In [ ]:
# ============================================================
# Cell 9 — HTML Visualization: Original vs Corrected (10+ images)
# ============================================================
# Generates an interactive HTML report showing:
#   - Image thumbnail
#   - Original vs enriched caption (diff highlighted)
#   - Extracted triples + type classification
#   - Hallucinated triples + reasoning
#   - KB evidence used for each decision

import difflib
import html as html_mod
from IPython.display import display, HTML as IPyHTML

HTML_PATH = f"{SAVE_DIR}/blip2_qualitative.html"

def word_diff_html(original, corrected):
    """Generate highlighted word-level diff: red strikethrough for removed, green for added."""
    orig_words = original.split()
    corr_words = corrected.split()
    sm = difflib.SequenceMatcher(None, orig_words, corr_words)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == 'equal':
            parts.append(' '.join(orig_words[i1:i2]))
        elif op == 'replace':
            parts.append(f'<span style="background:#ffcccc;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
            parts.append(f'<span style="background:#ccffcc;font-weight:bold;color:#060">{" ".join(corr_words[j1:j2])}</span>')
        elif op == 'delete':
            parts.append(f'<span style="background:#ffcccc;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
        elif op == 'insert':
            parts.append(f'<span style="background:#ccffcc;font-weight:bold;color:#060">{" ".join(corr_words[j1:j2])}</span>')
    return ' '.join(parts)

def img_to_b64_thumb(pil_img, max_size=400):
    """Convert PIL image to base64 thumbnail for HTML embedding."""
    img_copy = pil_img.copy()
    img_copy.thumbnail((max_size, max_size))
    buf = BytesIO()
    img_copy.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()

# ── Select images for visualization ──
# Priority: modified images first (sorted by edit rate), then unchanged
modified_imgs = [(i, v) for i, v in enriched_only.items()
                 if v["status"] == "modified" and i in images]
modified_imgs.sort(key=lambda x: -x[1]["edit_rate"])

unchanged_imgs = [(i, v) for i, v in enriched_only.items()
                  if v["status"] == "unchanged" and i in images]

# Take at least 10: prefer modified, pad with unchanged
viz_images = modified_imgs[:10]
if len(viz_images) < 10:
    viz_images += unchanged_imgs[:10 - len(viz_images)]

print(f"Generating HTML for {len(viz_images)} images "
      f"({sum(1 for _,v in viz_images if v['status']=='modified')} modified, "
      f"{sum(1 for _,v in viz_images if v['status']=='unchanged')} unchanged)...")

# ── Build HTML ──
CSS = """
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
       max-width: 1100px; margin: 0 auto; padding: 20px; background: #f5f5f5; }
h1 { color: #1a1a2e; border-bottom: 3px solid #16213e; padding-bottom: 10px; }
.image-card { background: white; border-radius: 12px; padding: 24px;
              margin-bottom: 30px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }
.image-header { display: flex; gap: 20px; margin-bottom: 16px; }
.thumb { border-radius: 8px; max-width: 350px; height: auto; }
.meta { flex: 1; }
.meta h2 { margin: 0 0 8px 0; color: #16213e; font-size: 18px; }
.status-modified { color: #e94560; font-weight: bold; }
.status-unchanged { color: #0f3460; font-weight: bold; }
.caption-box { background: #fafafa; border-left: 4px solid #ccc;
               padding: 12px 16px; margin: 8px 0; border-radius: 0 8px 8px 0;
               font-size: 14px; line-height: 1.6; }
.caption-box.original { border-left-color: #0f3460; }
.caption-box.enriched { border-left-color: #e94560; }
.caption-box.diff { border-left-color: #f0a500; background: #fffef0; }
.label { font-weight: bold; font-size: 12px; text-transform: uppercase;
         letter-spacing: 1px; margin-bottom: 4px; }
.label.orig { color: #0f3460; }
.label.enr { color: #e94560; }
.label.diff { color: #f0a500; }
.triple-table { width: 100%; border-collapse: collapse; margin: 12px 0; font-size: 13px; }
.triple-table th { background: #16213e; color: white; padding: 8px 12px;
                   text-align: left; font-size: 12px; }
.triple-table td { padding: 8px 12px; border-bottom: 1px solid #eee; }
.triple-table tr:hover { background: #f0f8ff; }
.hallucinated { background: #fff0f0 !important; }
.hallucinated td { color: #900; }
.badge { display: inline-block; padding: 2px 8px; border-radius: 12px;
         font-size: 11px; font-weight: bold; }
.badge-spatial { background: #e8f4fd; color: #0a6abf; }
.badge-action { background: #fef3e2; color: #b45309; }
.badge-attribute { background: #f0e8fd; color: #7c3aed; }
.badge-other { background: #e8e8e8; color: #555; }
.badge-high { background: #fee; color: #c00; }
.badge-medium { background: #fff3cd; color: #856404; }
.badge-low { background: #e8e8e8; color: #555; }
.kb-section { background: #f8f9fa; border-radius: 8px; padding: 12px 16px;
              margin: 12px 0; font-size: 13px; }
.kb-section summary { cursor: pointer; font-weight: bold; color: #16213e; }
.kb-facts { margin: 8px 0; padding-left: 16px; color: #444; }
.rpope-box { display: inline-block; padding: 6px 14px; border-radius: 8px;
             font-size: 13px; margin: 4px; }
.rpope-improved { background: #d4edda; color: #155724; }
.rpope-regressed { background: #f8d7da; color: #721c24; }
.rpope-same { background: #e2e3e5; color: #383d41; }
.summary-bar { display: flex; gap: 20px; margin: 20px 0; flex-wrap: wrap; }
.summary-stat { background: white; border-radius: 8px; padding: 16px 24px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1); text-align: center; }
.summary-stat .num { font-size: 28px; font-weight: bold; color: #16213e; }
.summary-stat .lbl { font-size: 12px; color: #666; text-transform: uppercase; }
"""

html_parts = []
html_parts.append(f'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="utf-8">\n'
                  f'<title>RelCheck BLIP-2 Qualitative Analysis</title>\n'
                  f'<style>{CSS}</style>\n</head>\n<body>\n'
                  f'<h1>RelCheck &mdash; BLIP-2 Qualitative Analysis</h1>')

# Summary stats
n_total = len(enriched_only)
n_mod = sum(1 for v in enriched_only.values() if v["status"] == "modified")
n_unch = n_total - n_mod

orig_pairs = results_by_method.get("blip2_orig", [])
enr_pairs = results_by_method.get("blip2_enriched_only", [])
orig_acc = sum(1 for p, g in orig_pairs if p == g) / max(len(orig_pairs), 1)
enr_acc = sum(1 for p, g in enr_pairs if p == g) / max(len(enr_pairs), 1)

html_parts.append(
    f'<div class="summary-bar">'
    f'<div class="summary-stat"><div class="num">{n_total}</div><div class="lbl">Images Tested</div></div>'
    f'<div class="summary-stat"><div class="num">{n_mod}</div><div class="lbl">Modified</div></div>'
    f'<div class="summary-stat"><div class="num">{n_unch}</div><div class="lbl">Unchanged</div></div>'
    f'<div class="summary-stat"><div class="num">{orig_acc:.1%}</div><div class="lbl">R-POPE Original</div></div>'
    f'<div class="summary-stat"><div class="num">{enr_acc:.1%}</div><div class="lbl">R-POPE Enriched</div></div>'
    f'</div>'
)

# ── Per-image cards ──
SPATIAL_KW = {"on","in","under","above","below","left","right","behind","front",
              "near","next","between","inside","outside","beside","top","bottom"}
ACTION_KW = {"holding","riding","wearing","eating","sitting","standing","walking",
             "running","playing","carrying","reading","looking","using","driving"}

for img_id, v in viz_images:
    original = v.get("original", blip2_captions.get(img_id, ""))
    corrected = v.get("corrected", original)
    status = v["status"]
    edit_rate = v.get("edit_rate", 0)
    errors = v.get("errors", [])
    kb = knowledge_bases.get(img_id, {})

    # Image thumbnail
    if img_id in images:
        b64_thumb = img_to_b64_thumb(images[img_id])
        img_tag = f'<img class="thumb" src="data:image/jpeg;base64,{b64_thumb}" alt="{img_id}">'
    else:
        img_tag = '<div style="width:350px;height:250px;background:#ddd;border-radius:8px"></div>'

    status_class = "status-modified" if status == "modified" else "status-unchanged"

    # R-POPE per-image scores
    orig_correct = img_orig.get(img_id, [])
    enr_correct = img_enr.get(img_id, [])
    orig_score = sum(orig_correct) / max(len(orig_correct), 1) if orig_correct else None
    enr_score = sum(enr_correct) / max(len(enr_correct), 1) if enr_correct else None

    rpope_html = ""
    if orig_score is not None and enr_score is not None:
        if enr_score > orig_score:
            cls = "rpope-improved"
        elif enr_score < orig_score:
            cls = "rpope-regressed"
        else:
            cls = "rpope-same"
        rpope_html = f'<span class="rpope-box {cls}">R-POPE: {orig_score:.0%} &rarr; {enr_score:.0%}</span>'

    html_parts.append(
        f'<div class="image-card">'
        f'<div class="image-header">{img_tag}'
        f'<div class="meta"><h2>{html_mod.escape(img_id)}</h2>'
        f'<p><span class="{status_class}">{status.upper()}</span>'
        f' &nbsp; edit_rate={edit_rate:.3f}'
        f' &nbsp; errors={len(errors)}'
        f' {rpope_html}</p></div></div>'
        f'<div class="label orig">Original Caption</div>'
        f'<div class="caption-box original">{html_mod.escape(original)}</div>'
        f'<div class="label enr">Enriched Caption</div>'
        f'<div class="caption-box enriched">{html_mod.escape(corrected)}</div>'
    )

    # Word diff (only if modified)
    if status == "modified":
        diff_html = word_diff_html(original, corrected)
        html_parts.append(
            f'<div class="label diff">Diff (red=removed, green=added)</div>'
            f'<div class="caption-box diff">{diff_html}</div>'
        )

    # Errors table
    if errors:
        rows = '<tr><th>Hallucinated Claim</th><th>Type</th><th>Confidence</th><th>Reasoning</th></tr>'
        for e in errors:
            claim = html_mod.escape(e.get("claim", ""))
            conf = e.get("confidence", "?")
            reason = html_mod.escape(e.get("reason", "")[:200])
            claim_lower = claim.lower()
            if any(kw in claim_lower for kw in SPATIAL_KW):
                type_badge = '<span class="badge badge-spatial">SPATIAL</span>'
            elif any(kw in claim_lower for kw in ACTION_KW):
                type_badge = '<span class="badge badge-action">ACTION</span>'
            else:
                type_badge = '<span class="badge badge-attribute">ATTRIBUTE</span>'
            conf_badge = f'<span class="badge badge-{conf.lower()}">{conf}</span>'
            rows += (f'<tr class="hallucinated"><td>{claim}</td>'
                     f'<td>{type_badge}</td><td>{conf_badge}</td><td>{reason}</td></tr>')
        html_parts.append(f'<table class="triple-table">{rows}</table>')
    else:
        html_parts.append('<p style="color:#666;font-style:italic">No errors detected &mdash; caption passed verification.</p>')

    # KB evidence (collapsible)
    hard = kb.get("hard_facts", [])
    spatial = kb.get("spatial_facts", [])
    visual = kb.get("visual_description", "")
    html_parts.append(
        f'<details class="kb-section">'
        f'<summary>Visual Knowledge Base ({len(hard)} objects, {len(spatial)} spatial facts)</summary>'
        f'<div class="kb-facts">'
        f'<strong>Detected Objects:</strong> {", ".join(hard[:15]) if hard else "None"}<br>'
        f'<strong>Spatial Facts:</strong> {"; ".join(spatial[:8]) if spatial else "None"}<br>'
        f'<strong>VLM Description:</strong> {html_mod.escape((visual or "None")[:300])}'
        f'</div></details>'
    )

    html_parts.append('</div>')

html_parts.append(
    '<footer style="text-align:center;color:#999;padding:20px;font-size:12px">'
    'Generated by RelCheck &mdash; CS298 Master&#39;s Project &mdash; Siddhi Patil, SJSU 2026'
    '</footer></body></html>'
)

# ── Save HTML ──
full_html = '\n'.join(html_parts)
with open(HTML_PATH, 'w') as f:
    f.write(full_html)

print(f"Saved qualitative report: {HTML_PATH}")
print(f"   File size: {len(full_html)/1024:.0f} KB")
print(f"   Images: {len(viz_images)}")

# Display inline in Colab
display(IPyHTML(f'<div style="max-height:800px;overflow-y:auto">{full_html}</div>'))
